# Week 06 · PyTorch：训练、验证与独立推理

Tensor 有 shape、dtype、device。线性层接收 [batch, features]，输出 [batch, classes]。CrossEntropyLoss 接收原始 logits 和整数类别，不要先做 softmax 再传入；它内部完成稳定的 log-softmax。反向传播计算梯度，optimizer.step 更新参数，zero_grad 清除旧梯度；漏掉清零会意外累积。

train() 与 eval() 改变 Dropout/BatchNorm 行为，不会关闭梯度。no_grad() 关闭梯度记录，降低推理内存。Dataset 描述样本，DataLoader 负责批处理和打乱；验证集不用于反向更新。保存验证损失最低的 state_dict，加载到同样结构的模型。

为保证本地立即运行，使用 sklearn 自带 8×8 手写数字图像，CPU 训练小型 MLP，不要求下载 Fashion-MNIST。这个教学实验不代表大规模图像系统。训练、验证、测试三份严格分开。独立推理脚本只加载模型、输入特征并输出预测。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from pathlib import Path
import copy

torch.manual_seed(42);torch.set_num_threads(2)
digits=load_digits()
X=(digits.data/16).astype("float32");y=digits.target.astype("int64")
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
X_valid,X_test,y_valid,y_test=train_test_split(X_temp,y_temp,test_size=.5,stratify=y_temp,random_state=42)
loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train)),batch_size=64,shuffle=True)
def make_model():return nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Dropout(.1),nn.Linear(64,10))
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=make_model().to(device)
loss_function=nn.CrossEntropyLoss();optimizer=torch.optim.Adam(model.parameters(),lr=.01)
history=[];best=float("inf");best_state=None
for epoch in range(12):
    model.train();total=0
    for features,labels in loader:
        features,labels=features.to(device),labels.to(device)
        optimizer.zero_grad()
        loss=loss_function(model(features),labels)
        loss.backward();optimizer.step()
        total+=loss.item()*len(features)
    model.eval()
    with torch.no_grad():
        validation=loss_function(model(torch.tensor(X_valid).to(device)),torch.tensor(y_valid).to(device)).item()
    history.append((total/len(X_train),validation))
    if validation<best:
        best=validation;best_state=copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
torch.save({k:v.cpu() for k,v in best_state.items()},"week06-best.pt")
restored=make_model();restored.load_state_dict(torch.load("week06-best.pt",weights_only=True));restored.eval()
with torch.no_grad():
    predictions=restored(torch.tensor(X_test)).argmax(1).numpy()
print("测试准确率：",float((predictions==y_test).mean()),"设备：",device)
plt.plot(history);plt.legend(["train loss","validation loss"]);plt.xlabel("epoch");plt.show()
# 保存一个真正可独立运行的推理入口，而不是依赖当前 Notebook 的变量。
script="""import torch, json, sys
from torch import nn
model=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Dropout(.1),nn.Linear(64,10))
model.load_state_dict(torch.load("week06-best.pt",map_location="cpu",weights_only=True))
model.eval()
features=torch.tensor(json.load(open(sys.argv[1])),dtype=torch.float32).reshape(-1,64)
with torch.no_grad(): print(model(features).argmax(1).tolist())
"""
Path("week06-infer.py").write_text(script,encoding="utf-8")

## 练习 / Exercises
移除 Dropout 后比较曲线。解释 eval 与 no_grad 的不同；用导出的脚本预测一个样本。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
import json,subprocess,sys
Path("week06-input.json").write_text(json.dumps(X_test[:1].tolist()))
result=subprocess.run([sys.executable,"week06-infer.py","week06-input.json"],capture_output=True,text=True,check=True)
assert int(json.loads(result.stdout)[0])==int(predictions[0])
print("独立进程推理：",result.stdout)